[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/09_naip-imagery/09_03_classify_naip_landcover.ipynb)

# Module 9, Lesson 3: Build and Check a Simple NAIP Land Cover Classification
## A screening map from a small labeled sample, and an honest look at where it fails

### Welcome!

Early in a drainage study you often need a quick answer to "how much of this area is paved or built on, and how much is still vegetated?" before the detailed survey or the county's impervious layer arrives. A 0.6 m aerial image has far more detail than Annual NLCD's 30 m cells. But an image is just colors. To turn it into a map of classes, you have to tell the computer what each class looks like, and then check whether it learned the right thing.

In this lesson you'll build a four-class screening map of the Lesson 1 teaching area: **vegetation, bare soil, built/paved, and water**. You'll train a Random Forest classifier from hand-drawn training polygons, then test it against a *separate* set of validation points the model never sees. It'll get a lot right. It'll also get one thing badly wrong, and understanding why is the most useful part of the lesson.

### What You'll Work Through Today

- Understand supervised classification: features, labels, training, prediction, and validation
- Load reviewed training polygons and independent validation points, and check that they really are independent
- Extract a balanced, capped sample of training pixels
- Train a Random Forest on a standard Colab CPU
- Predict every pixel in manageable chunks and save a georeferenced class raster
- Check the map against independent points with a confusion matrix
- Look at the failures up close and explain them
- Summarize class areas correctly, excluding NoData
- Put the result in context with Module 8's Annual NLCD
- Find and fix a validation mistake that makes a mediocre model look perfect

### Lesson Structure

1. **Mental Model** - How supervised classification works, in plain terms
2. **Workspace Setup** - Packages and settings
3. **Load and Check the Inputs** - Image, training polygons, validation points
4. **See the Labels** - Where the model's knowledge comes from
5. **Extract Training Samples** - Balanced and capped
6. **Train the Classifier** - A small Random Forest
7. **Predict and Save the Map** - Chunked prediction and a GeoTIFF
8. **Validate Against Independent Points** - Confusion matrix and class accuracy
9. **Look at the Failures** - Image chips of every miss
10. **Class Areas** - Valid pixels only
11. **Context from Annual NLCD** - A comparison, not a truth test
12. **Practice Debugging** - A validation score that's too good to be true
13. **Record the Limitations** - A summary that travels with the map

## Lesson Information

**Lesson Level:** Advanced

**Purpose:** Produce a documented screening-level land cover map from NAIP, validated against independent labels, with its limits stated plainly.

### Learning Objectives

By the end of this lesson, you'll be able to:

- Define a small land cover legend and explain the difference between training labels and independent validation labels.
- Extract red, green, blue, and NIR features from labeled pixels while excluding masked pixels.
- Train a reproducible Random Forest classifier and predict a raster in chunks.
- Validate a classification with a confusion matrix, producer's accuracy, and user's accuracy, and explain what a small validation sample can and can't support.
- Export a georeferenced class raster and compute class areas over valid pixels only.
- Recognize validation leakage and explain why it inflates accuracy.

### Prerequisites

- Lessons 1 and 2 of this module, or at least their mental models. You don't need to have run them; this notebook gets its own data.
- Module 4 (raster basics) and Module 8 (categorical rasters and class-area summaries)
- Comfort with NumPy arrays and pandas tables

### Inputs and Outputs

| | File | Notes |
|---|---|---|
| Input | `naip_2022_teaching_aoi.tif` | From Lesson 1 or the course copy |
| Input | `naip_training_polygons.geojson` | 13 polygons, 4 classes, drawn on the 2022 image |
| Input | `naip_validation_points.geojson` | 48 points, at least 30 m from any training polygon |
| Input (optional) | `annual_nlcd_2025_land_cover.tif` | Module 8 course file, used for context in Part 11 |
| Output | `module9_outputs/naip_2022_landcover_screening.tif` | Single-band class raster with a color table |
| Output | `module9_outputs/naip_landcover_screening_map.png` | Map figure with legend |
| Output | `module9_outputs/naip_validation_results.csv` | Every validation point, reference vs predicted |
| Output | `module9_outputs/naip_confusion_matrix.csv` | Counts by reference and predicted class |
| Output | `module9_outputs/naip_landcover_class_area.csv` | Area and percent by class |
| Output | `module9_outputs/naip_classification_summary.txt` | Source, method, accuracy, and limitations |

Everything runs on a standard Colab CPU in a couple of minutes. No GPU, deep learning package, or pretrained model is needed.

## Using AI in This Lesson

Machine learning code is easy to write and easy to get subtly wrong. The mistakes that matter in this lesson don't raise errors: labels leaking into validation, class codes getting mixed up, and NoData sneaking into area totals. Use your AI assistant to **trace data**, not to generate models.

Questions worth trying:

- *"Trace where the training labels come from in my notebook and where the validation labels come from. Is there any path where a validation location could end up in the training data?"*
- *"Here's my confusion matrix. Explain producer's accuracy and user's accuracy for the water class in plain words."*
- *"My classifier labels a pond as built/paved. List possible causes, starting with the training data."*

Don't let the assistant invent class names, move labels to improve a score, or describe the map as "accurate" without pointing to the validation table.

## Part 1: Mental Model

### Supervised Classification in Plain Terms

Think of training a new technician to sort soil samples. You hand them labeled examples ("this is clay, this is sand"), they learn what distinguishes each type, and then they sort a pile of unlabeled samples. To find out how good they are, you give them a few samples *you* already know and they haven't seen before, and you check their answers.

Image classification works the same way:

| Step | Soil technician | This lesson |
|---|---|---|
| Features | Color, texture, grit | Red, green, blue, NIR values of each pixel |
| Labels | "Clay", "sand" | Class codes 1 to 4 |
| Training | Studying labeled samples | Fitting a Random Forest to pixels inside training polygons |
| Prediction | Sorting the pile | Classifying every pixel in the image |
| Validation | Blind test samples | Checking predictions at separate validation points |

### Why Validation Has to Be Independent

If the "blind test" uses samples the technician already studied, a high score just proves they have a good memory. Pixels are the same, with one extra twist: neighboring pixels look almost identical. A validation point in the middle of a training polygon, or right next to one, is barely a new test. The labels for this lesson keep every validation point at least 30 m from any training polygon, and the notebook checks that.

### What a Random Forest Is

A decision tree asks a series of yes/no questions: "Is NIR above 180? Is red below 90?" and ends at a class. A single tree tends to memorize its training data. A Random Forest grows many trees, each from a random subset of the training pixels and features, and lets them vote. It's fast on a CPU, needs little tuning, and is a common baseline for classifying imagery.

### Reading a Confusion Matrix

Rows are the **reference** class (what the validation label says). Columns are the **predicted** class (what the model said). The diagonal holds correct answers.

- **Producer's accuracy** (row view): of the points that really are water, what share did the map call water? Low values mean the map *misses* that class.
- **User's accuracy** (column view): of the points the map called water, what share really are water? Low values mean the map *over-reports* that class.

### Screening Map, Not Design Data

A built/paved class is a visual land cover class. It isn't percent impervious, it isn't connected impervious area, and it isn't a calibrated curve number or roughness map. Keep that in mind when the area table comes out at the end.

### The Legend

| Code | Class | Includes | Excludes or treats with caution |
|---|---|---|---|
| 1 | Vegetation | Irrigated turf, dry rangeland grass, shrubs, trees | Shadows under trees |
| 2 | Bare soil | Graded construction ground, dirt roads, exposed soil pads | Gravel lots that look like pavement |
| 3 | Built/paved | Roofs, asphalt, concrete, parked vehicles | Not the same as impervious area |
| 4 | Water | Open water in ponds | Wet soil, shaded water edges |
| 0 | NoData | Pixels outside valid imagery | Never counted in area totals |

Four classes is deliberately modest. The more classes you add, the more labels you need, and the harder it is to validate each one.

## Part 2: Workspace Setup

Colab already includes scikit-learn. The install check covers Rasterio and GeoPandas in case you're running somewhere else.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {'rasterio': 'rasterio', 'geopandas': 'geopandas', 'sklearn': 'scikit-learn'}
missing = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Done.")
else:
    print("All required packages are already installed.")

In [ ]:
# Standard library
import datetime
import math
import shutil
from pathlib import Path

# Tables, arrays, and web requests
import numpy as np
import pandas as pd
import requests

# Vector and raster data
import geopandas as gpd
from shapely.geometry import box
import rasterio
import rasterio.errors
from rasterio.features import geometry_mask, rasterize
from rasterio.plot import plotting_extent
from rasterio.windows import Window, from_bounds

# Machine learning
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

# Plotting
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Every figure in this lesson uses Arial. Colab runs on Linux, which does not
# ship Arial, so the list falls back to Liberation Sans: a font drawn with the
# same letter widths and shapes, made as a drop-in replacement. Naming a font
# that does exist is also what keeps matplotlib from printing a warning for
# every label it draws.
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = FONT_PREFERENCE

installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
active_font = next((name for name in FONT_PREFERENCE if name in installed_fonts), None)
print(f"Figure font: {active_font or 'matplotlib default'}")

for package in (np, pd, gpd, rasterio, sklearn):
    print(f"{package.__name__:<12} {package.__version__}")

### Set the Lesson Controls

The random seed makes the sampling and the Random Forest repeatable. Change it and you'll get slightly different trees and possibly a slightly different score. That's a useful reminder that one accuracy number from one run isn't a property of the method.

In [ ]:
AOI_CRS = 'EPSG:26913'
AOI_BOUNDS = (517750, 4549350, 518850, 4550900)
REFERENCE_YEAR = 2022
BAND_NAMES = ('Red', 'Green', 'Blue', 'Near infrared')

# Label files and the minimum training-to-validation distance they must honor
TRAINING_FILE = 'naip_training_polygons.geojson'
VALIDATION_FILE = 'naip_validation_points.geojson'
MIN_SEPARATION_M = 30

# Sampling and model settings
MAX_PIXELS_PER_CLASS = 2000
RANDOM_SEED = 42
N_TREES = 100

# The legend used throughout the lesson
CLASSES = pd.DataFrame([
    {'class_id': 1, 'class_name': 'Vegetation', 'color': '#4d8b31'},
    {'class_id': 2, 'class_name': 'Bare soil', 'color': '#d9b26f'},
    {'class_id': 3, 'class_name': 'Built/paved', 'color': '#6b6b6b'},
    {'class_id': 4, 'class_name': 'Water', 'color': '#2f6db5'},
])
CLASS_IDS = CLASSES['class_id'].tolist()
CLASS_NAME = dict(zip(CLASSES['class_id'], CLASSES['class_name']))
CLASS_COLORMAP = ListedColormap(CLASSES['color'].tolist())

COURSE_DATA_URL = (
    'https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/'
    'main/notebooks/09_naip-imagery/data'
)
MODULE8_DATA_URL = (
    'https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/'
    'main/notebooks/08_landuse-data/data'
)

OUTPUT_DIR = Path('module9_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
CLASSES

## Part 3: Load and Check the Inputs

### Get the Files

The helper looks for each file locally first (your Lesson 1 output, or a cloned course repository) and downloads the course copy if it isn't there. It prints where each file came from.

In [ ]:
def get_course_file(filename, base_url=COURSE_DATA_URL, module_folder='09_naip-imagery'):
    """Return a local path for a course file, downloading it only if needed."""
    destination = OUTPUT_DIR / filename
    for candidate in [destination, Path(filename), Path('data') / filename,
                      Path(f'notebooks/{module_folder}/data') / filename,
                      Path(f'../{module_folder}/data') / filename]:
        if candidate.exists():
            if candidate.resolve() != destination.resolve():
                shutil.copyfile(candidate, destination)
            return destination, f"local file ({candidate})"
    response = requests.get(f"{base_url}/{filename}", timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination, "course repository download"


image_path, image_source = get_course_file(f'naip_{REFERENCE_YEAR}_teaching_aoi.tif')
training_path, training_source = get_course_file(TRAINING_FILE)
validation_path, validation_source = get_course_file(VALIDATION_FILE)

for label, path, source in [('Image', image_path, image_source),
                            ('Training polygons', training_path, training_source),
                            ('Validation points', validation_path, validation_source)]:
    print(f"{label:<18} {path}  <- {source}")

### Check the Image

Same idea as Lessons 1 and 2: confirm the file before using it. The cell also reads the whole image into memory. At 1,834 by 2,584 pixels and four bands, that's about 18 MB, which is comfortable in Colab.

In [ ]:
with rasterio.open(image_path) as source:
    image = source.read()                        # (4, rows, cols) uint8
    valid_mask = source.dataset_mask() > 0       # True where imagery is valid
    image_transform = source.transform
    image_crs = source.crs
    image_extent = plotting_extent(source)
    image_bounds = source.bounds
    image_tags = source.tags()
    descriptions = source.descriptions

problems = []
if image.shape[0] != 4 or image.dtype != np.uint8:
    problems.append(f"expected 4 uint8 bands, found {image.shape[0]} {image.dtype}")
if image_crs is None or image_crs.to_string() != AOI_CRS:
    problems.append(f"expected {AOI_CRS}, found {image_crs}")
if tuple(descriptions) != BAND_NAMES:
    problems.append(f"unexpected band order {descriptions}")
if image_tags.get('naip_year') != str(REFERENCE_YEAR):
    problems.append(f"expected NAIP {REFERENCE_YEAR}, found {image_tags.get('naip_year')}")
if problems:
    raise RuntimeError("Image check failed:\n- " + "\n- ".join(problems))

cell_area_m2 = abs(image_transform.a * image_transform.e - image_transform.b * image_transform.d)
print(f"NAIP {image_tags['naip_year']}, acquired {image_tags['acquisition_date']}, "
      f"item {image_tags['source_item_id']}")
print(f"{image.shape[2]:,} x {image.shape[1]:,} pixels at {image_transform.a} m "
      f"({cell_area_m2:.2f} m² per pixel)")
print(f"Valid pixels: {valid_mask.mean():.2%}")

### Check the Labels

The label files are stored in longitude and latitude (EPSG:4326), which is the GeoJSON standard. The first job is reprojecting them to the image CRS. Then the cell checks:

- Every class code is in the legend, and every class appears in both files
- Training features are polygons; validation features are points
- Every label falls inside the image
- Every validation point is at least 30 m from every training polygon

That last check is the one that protects the validation score. It's cheap, so run it every time the labels change.

In [ ]:
training = gpd.read_file(training_path)
validation = gpd.read_file(validation_path)
if training.crs is None or validation.crs is None:
    raise RuntimeError("A label file has no CRS.")

print(f"Label files are stored in {training.crs}; reprojecting to {image_crs}.")
training = training.to_crs(image_crs)
validation = validation.to_crs(image_crs)

image_box = box(*image_bounds)
distance_to_training = validation.geometry.distance(training.geometry.union_all())
validation['distance_to_training_m'] = distance_to_training.round(1)

label_checks = [
    ('Training class codes are in the legend', set(training['class_id']) <= set(CLASS_IDS)),
    ('Validation class codes are in the legend', set(validation['class_id']) <= set(CLASS_IDS)),
    ('Every class has training polygons', set(training['class_id']) == set(CLASS_IDS)),
    ('Every class has validation points', set(validation['class_id']) == set(CLASS_IDS)),
    ('Training features are polygons', training.geom_type.isin(['Polygon', 'MultiPolygon']).all()),
    ('Validation features are points', (validation.geom_type == 'Point').all()),
    ('All labels are inside the image',
     training.within(image_box).all() and validation.within(image_box).all()),
    (f'Validation points are at least {MIN_SEPARATION_M} m from training',
     (distance_to_training >= MIN_SEPARATION_M).all()),
]
label_check_table = pd.DataFrame(label_checks, columns=['check', 'passed'])
label_check_table['status'] = np.where(label_check_table['passed'], 'PASS', 'FAIL')
display(label_check_table[['check', 'status']])
if not label_check_table['passed'].all():
    raise RuntimeError("A label check failed. Fix the label files before training.")

label_counts = pd.DataFrame({
    'training_polygons': training.groupby('class_id').size(),
    'training_area_m2': training.assign(area=training.area).groupby('class_id')['area'].sum().round(0),
    'validation_points': validation.groupby('class_id').size(),
}).rename(index=CLASS_NAME)
print(f"\nClosest validation point to any training polygon: {distance_to_training.min():.1f} m")
print(f"Label review status: {training['review_status'].iloc[0]}")
label_counts

A few things to notice in that table:

- **The classes aren't balanced.** Vegetation covers roughly 40,000 m² of training polygons and water only about 2,000 m². Part 5 samples the same number of pixels from each class so the model doesn't simply learn "most things are vegetation."
- **48 validation points is a small sample.** It's enough to find obvious failures, but any accuracy number from it has a wide margin of error. Part 8 shows how wide.
- **The review status field.** Labels are data, and data has provenance. Before teaching with these labels, or reusing them for anything, someone should review each polygon and point against the imagery.

## Part 4: See the Labels

Everything the model learns comes from the training polygons, so it's worth looking at exactly where they are. Validation points are drawn as small circles. Pay attention to *which* ponds, fields, and roofs are used for training and which are left for validation.

In [ ]:
def percentile_stretch(bands, lower=2, upper=98):
    """Stretch a (3, rows, cols) array to 0-1 for display. The data are not changed."""
    display_array = np.zeros(bands.shape, dtype='float32')
    for index in range(bands.shape[0]):
        band = bands[index].astype('float32')
        low, high = np.percentile(band, [lower, upper])
        display_array[index] = np.clip((band - low) / max(high - low, 1), 0, 1)
    return np.moveaxis(display_array, 0, -1)


true_color = percentile_stretch(image[[0, 1, 2]])

fig, ax = plt.subplots(figsize=(10, 13))
ax.imshow(true_color, extent=image_extent)
for class_id, color in zip(CLASSES['class_id'], CLASSES['color']):
    training[training['class_id'] == class_id].boundary.plot(ax=ax, color=color, linewidth=2.5)
    validation[validation['class_id'] == class_id].plot(
        ax=ax, color=color, markersize=45, edgecolor='white', linewidth=1)
handles = [Patch(facecolor=color, edgecolor='white', label=f"{cid} {name}")
           for cid, name, color in CLASSES[['class_id', 'class_name', 'color']].itertuples(index=False)]
ax.legend(handles=handles, loc='lower right', title='Outlines: training   Dots: validation')
ax.set_title(f"Training polygons and validation points on NAIP {REFERENCE_YEAR}")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

Before you move on, look at the water labels. The only water training polygon is in the east lobe of the long **north pond**. The validation points for water are in two places: other parts of the north pond, and the smaller **campus pond** on the west side. Keep that in mind when you get to Part 8.

**Question to ask your AI assistant:** *"My training polygons for water all come from one pond. What could go wrong when the classifier meets a second pond with different depth, sediment, or shading?"*

## Part 5: Extract Training Samples

### From Polygons to Pixels

The model trains on pixels, not polygons. The next cell:

1. **Rasterizes** the training polygons onto the image grid. Each pixel whose center falls inside a polygon gets that polygon's class code; everything else gets 0.
2. **Keeps only valid pixels.** Masked pixels are never used.
3. **Samples up to `MAX_PIXELS_PER_CLASS` pixels per class** at random, with a fixed seed. The cap keeps classes balanced and the lesson fast.

The result is a feature table `X_train` (one row per pixel, one column per band) and a label array `y_train`.

In [ ]:
training_raster = rasterize(
    ((geometry, int(class_id)) for geometry, class_id in zip(training.geometry, training['class_id'])),
    out_shape=valid_mask.shape, transform=image_transform, fill=0, dtype='uint8',
)

rng = np.random.default_rng(RANDOM_SEED)
feature_blocks, label_blocks, sample_rows = [], [], []
for class_id in CLASS_IDS:
    rows, cols = np.nonzero((training_raster == class_id) & valid_mask)
    n_sample = min(MAX_PIXELS_PER_CLASS, rows.size)
    pick = rng.choice(rows.size, size=n_sample, replace=False)
    feature_blocks.append(image[:, rows[pick], cols[pick]].T)
    label_blocks.append(np.full(n_sample, class_id))
    sample_rows.append({'class_name': CLASS_NAME[class_id],
                        'labeled_pixels_available': rows.size,
                        'pixels_sampled': n_sample})

X_train = np.vstack(feature_blocks).astype('float32')
y_train = np.concatenate(label_blocks)

print(f"Training table: {X_train.shape[0]:,} pixels x {X_train.shape[1]} features {BAND_NAMES}")
pd.DataFrame(sample_rows)

Water has the fewest labeled pixels available, but still more than the cap, so every class ends up with 2,000 samples.

Balanced sampling fixes the *count* problem. It doesn't fix the *variety* problem: 2,000 pixels from one pond are still 2,000 examples of one pond.

## Part 6: Train the Classifier

The Random Forest settings are modest on purpose:

- `n_estimators=100` trees is plenty for four features and four classes.
- `min_samples_leaf=2` stops individual trees from memorizing single pixels.
- `random_state` makes the result repeatable.
- `n_jobs=-1` uses all available CPU cores.

Training takes a few seconds.

In [ ]:
model = RandomForestClassifier(
    n_estimators=N_TREES, min_samples_leaf=2, random_state=RANDOM_SEED, n_jobs=-1,
)
model.fit(X_train, y_train)

importance = pd.DataFrame({'feature': BAND_NAMES, 'importance': model.feature_importances_})
importance.sort_values('importance', ascending=False).round(3)

Feature importance tells you which bands the trees leaned on most. Expect the near-infrared band near the top: it's what separates vigorous vegetation from everything else and water from dark pavement. The importance values describe *this* model on *these* labels, not a general truth about NAIP.

## Part 7: Predict and Save the Map

### Predict in Chunks

The image has about 4.7 million pixels. Predicting all of them at once works in Colab, but it builds a large temporary array. Predicting in blocks of rows keeps memory steady and scales to bigger images. Pixels outside the valid mask stay 0 (NoData).

In [ ]:
BLOCK_ROWS = 256
n_rows = valid_mask.shape[0]
classified = np.zeros(valid_mask.shape, dtype='uint8')   # 0 = NoData

for row_start in range(0, n_rows, BLOCK_ROWS):
    row_stop = min(row_start + BLOCK_ROWS, n_rows)
    block_valid = valid_mask[row_start:row_stop]
    pixels = image[:, row_start:row_stop][:, block_valid].T.astype('float32')
    if pixels.size:
        classified[row_start:row_stop][block_valid] = model.predict(pixels)

print(f"Classified {valid_mask.sum():,} valid pixels in {math.ceil(n_rows / BLOCK_ROWS)} blocks.")
print("Class codes present:", sorted(np.unique(classified).tolist()))

### Write a Georeferenced Class Raster

The class raster copies the image's transform and CRS exactly, so it overlays the imagery pixel for pixel in any GIS. It's a single `uint8` band with 0 as NoData, and it carries:

- A **color table**, so GIS software shows the legend colors automatically
- **Tags** recording the legend, source image, classifier, and screening-only status

In [ ]:
def hex_to_rgba(hex_color):
    """Convert '#rrggbb' to an (r, g, b, 255) tuple for a GeoTIFF color table."""
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4)) + (255,)


classified_path = OUTPUT_DIR / f'naip_{REFERENCE_YEAR}_landcover_screening.tif'
profile = {
    'driver': 'GTiff', 'count': 1, 'dtype': 'uint8', 'nodata': 0,
    'width': classified.shape[1], 'height': classified.shape[0],
    'crs': image_crs, 'transform': image_transform,
    'compress': 'deflate', 'tiled': True, 'blockxsize': 256, 'blockysize': 256,
}
with rasterio.open(classified_path, 'w', **profile) as destination:
    destination.write(classified, 1)
    destination.write_colormap(1, {0: (0, 0, 0, 0), **{
        cid: hex_to_rgba(color) for cid, color in zip(CLASSES['class_id'], CLASSES['color'])}})
    destination.set_band_description(1, 'Screening land cover class')
    destination.update_tags(
        class_legend='; '.join(f"{cid}={name}" for cid, name in CLASS_NAME.items()) + '; 0=NoData',
        source_image=image_path.name,
        source_item_id=image_tags['source_item_id'],
        acquisition_date=image_tags['acquisition_date'],
        classifier=f"scikit-learn {sklearn.__version__} RandomForestClassifier "
                   f"(n_estimators={N_TREES}, min_samples_leaf=2, random_state={RANDOM_SEED})",
        features='Red, Green, Blue, Near infrared (uint8)',
        training_labels=TRAINING_FILE,
        status='Screening map for review; not impervious area; not for design',
        created=datetime.date.today().isoformat(),
    )

# Reopen and confirm it lines up with the image
with rasterio.open(classified_path) as check:
    same_grid = (check.transform == image_transform and check.crs == image_crs
                 and check.shape == valid_mask.shape)
    values_ok = set(np.unique(check.read(1)).tolist()) <= {0, *CLASS_IDS}
if not (same_grid and values_ok):
    raise RuntimeError("The class raster does not match the image grid or has unexpected codes.")
print(f"Saved: {classified_path} (grid matches the image; codes are valid)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 11))
axes[0].imshow(true_color, extent=image_extent)
axes[0].set_title(f"NAIP {REFERENCE_YEAR} true color")
axes[1].imshow(np.ma.masked_equal(classified, 0), cmap=CLASS_COLORMAP, vmin=0.5, vmax=4.5,
               extent=image_extent, interpolation='nearest')
axes[1].set_title("Screening land cover (Random Forest, 4 bands)")
axes[1].legend(handles=[Patch(color=color, label=name) for name, color
                        in CLASSES[['class_name', 'color']].itertuples(index=False)],
               loc='lower right')
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.tight_layout()
map_path = OUTPUT_DIR / 'naip_landcover_screening_map.png'
fig.savefig(map_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved: {map_path}")

### First Impressions

Compare the two panels before looking at any numbers. When this lesson was prepared, the map picked up the road network, roundabouts, rooftops, and parking lots cleanly, and the athletic field and graded construction area were easy to spot. It also showed some patterns worth questioning:

- Parts of the dry rangeland came out as bare soil, especially where the grass is thin or trails cross it.
- Building shadows and dark roofs sometimes came out as water or built/paved.
- Look closely at the campus pond on the west side. What class did it get?

A map that *looks* right is a hypothesis. The next part tests it.

## Part 8: Validate Against Independent Points

### Compare Predictions With Reference Labels

For each validation point, the next cell reads the predicted class at that pixel and compares it with the reference label. The model never saw any of these pixels, or any pixel within 30 m of them, during training.

In [ ]:
point_rows, point_cols = rasterio.transform.rowcol(
    image_transform, validation.geometry.x.to_numpy(), validation.geometry.y.to_numpy())
point_rows, point_cols = np.asarray(point_rows), np.asarray(point_cols)

results = validation[['point_id', 'class_id', 'class_name', 'description',
                      'distance_to_training_m']].copy()
results = results.rename(columns={'class_id': 'reference_id', 'class_name': 'reference_class'})
results['predicted_id'] = classified[point_rows, point_cols]
results['predicted_class'] = results['predicted_id'].map(CLASS_NAME)
results['correct'] = results['reference_id'] == results['predicted_id']
results['easting'] = validation.geometry.x.round(1)
results['northing'] = validation.geometry.y.round(1)

if (results['predicted_id'] == 0).any():
    raise RuntimeError("A validation point falls on NoData. Check the labels and the mask.")

matrix = confusion_matrix(results['reference_id'], results['predicted_id'], labels=CLASS_IDS)
confusion = pd.DataFrame(matrix,
                         index=[f"Reference: {CLASS_NAME[c]}" for c in CLASS_IDS],
                         columns=[f"Predicted: {CLASS_NAME[c]}" for c in CLASS_IDS])
confusion

### Class-by-Class Accuracy

The next table turns the confusion matrix into the two per-class measures from Part 1, plus overall accuracy.

It also reports a **95 percent confidence interval** for overall accuracy using the Wilson method. With only 48 points, the interval is wide. That's the honest way to report a small validation sample.

In [ ]:
correct_per_class = np.diag(matrix)
reference_totals = matrix.sum(axis=1)
predicted_totals = matrix.sum(axis=0)

class_accuracy = pd.DataFrame({
    'class': [CLASS_NAME[c] for c in CLASS_IDS],
    'validation_points': reference_totals,
    'correct': correct_per_class,
    'producers_accuracy': np.round(correct_per_class / np.maximum(reference_totals, 1), 2),
    'mapped_as_this_class': predicted_totals,
    'users_accuracy': np.where(predicted_totals > 0,
                               np.round(correct_per_class / np.maximum(predicted_totals, 1), 2),
                               np.nan),
})


def wilson_interval(successes, total, z=1.96):
    """95% Wilson score interval for a proportion."""
    p = successes / total
    center = (p + z ** 2 / (2 * total)) / (1 + z ** 2 / total)
    margin = z * math.sqrt(p * (1 - p) / total + z ** 2 / (4 * total ** 2)) / (1 + z ** 2 / total)
    return center - margin, center + margin


n_points = len(results)
n_correct = int(results['correct'].sum())
overall_accuracy = n_correct / n_points
ci_low, ci_high = wilson_interval(n_correct, n_points)

print(f"Overall accuracy: {n_correct} of {n_points} points = {overall_accuracy:.1%}")
print(f"95% confidence interval: {ci_low:.1%} to {ci_high:.1%}")
class_accuracy

Read the table class by class, not just the overall number. When this lesson was prepared, overall accuracy came out around 83 percent, but that average hid a very uneven picture:

- **Built/paved** had high producer's accuracy. The map found nearly every roof, road, and parking lot point.
- **Water** had low producer's accuracy. Every point in the **campus pond** was labeled built/paved. The north pond points were fine.
- **Vegetation and bare soil** confused each other on a couple of points: dry, thin rangeland versus graded ground.

Your numbers may differ slightly with other package versions, but the pattern should be the same.

**Question to ask your AI assistant:** *"My water class has user's accuracy of 1.0 but producer's accuracy under 0.5. Explain what each number means for someone using this map to find ponds."*

## Part 9: Look at the Failures

Numbers tell you *that* something failed. Pictures tell you *why*. The next cell shows a 40 m chip around every misclassified validation point: true color on top and the predicted classes underneath, with the point marked.

In [ ]:
misses = results[~results['correct']].reset_index(drop=True)
print(f"Misclassified validation points: {len(misses)}")
display(misses[['point_id', 'reference_class', 'predicted_class', 'description']])

if not misses.empty:
    half = 20  # meters
    n_cols = min(len(misses), 6)
    n_rows = 2 * math.ceil(len(misses) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.9 * n_rows), squeeze=False)
    for axis in axes.ravel():
        axis.axis('off')
    for i, miss in misses.iterrows():
        top, col = 2 * (i // n_cols), i % n_cols
        r, c = rasterio.transform.rowcol(image_transform, miss['easting'], miss['northing'])
        size = int(round(half / image_transform.a))
        rows, cols = slice(max(r - size, 0), r + size), slice(max(c - size, 0), c + size)
        axes[top, col].imshow(true_color[rows, cols])
        axes[top, col].plot(size, size, marker='+', color='red', markersize=14, mew=2)
        axes[top, col].set_title(f"{miss['point_id']}: {miss['reference_class']}", fontsize=9)
        axes[top + 1, col].imshow(np.ma.masked_equal(classified[rows, cols], 0), cmap=CLASS_COLORMAP,
                                  vmin=0.5, vmax=4.5, interpolation='nearest')
        axes[top + 1, col].plot(size, size, marker='+', color='red', markersize=14, mew=2)
        axes[top + 1, col].set_title(f"predicted {miss['predicted_class']}", fontsize=9)
    plt.tight_layout()
    plt.show()

### Why the Campus Pond Failed

An average hides what matters here, which is the *range* of values the model saw for each class. The next cell compares four groups of pixels: the training water, the validation points in each pond, and the training asphalt parking lot. For red and NIR it reports each group's range from the 1st to the 99th percentile.

In [ ]:
def band_summary(pixels):
    """Average band values and 1st-99th percentile ranges for an (n, 4) pixel table."""
    red_low, red_high = np.percentile(pixels[:, 0], [1, 99])
    nir_low, nir_high = np.percentile(pixels[:, 3], [1, 99])
    return {
        'pixels': len(pixels),
        'mean R, G, B, NIR': tuple(int(v) for v in pixels.mean(axis=0).round(0)),
        'Red range': f"{red_low:.0f} to {red_high:.0f}",
        'NIR range': f"{nir_low:.0f} to {nir_high:.0f}",
    }


campus_pond = results['description'].str.contains('Campus pond').to_numpy()
north_pond = results['description'].str.contains('North pond').to_numpy()
asphalt_lot = training[training['description'].str.contains('Asphalt parking lot')]
asphalt_mask = rasterize(asphalt_lot.geometry, out_shape=valid_mask.shape,
                         transform=image_transform, fill=0, default_value=1).astype(bool)

pond_comparison = pd.DataFrame({
    'Training water (north pond, east lobe)': band_summary(X_train[y_train == 4]),
    'Validation points: north pond': band_summary(
        image[:, point_rows[north_pond], point_cols[north_pond]].T.astype('float32')),
    'Validation points: campus pond': band_summary(
        image[:, point_rows[campus_pond], point_cols[campus_pond]].T.astype('float32')),
    'Training asphalt parking lot': band_summary(
        image[:, asphalt_mask & valid_mask].T.astype('float32')),
}).T
pond_comparison

When this lesson was prepared, the training water was remarkably uniform: NIR from about 21 to 26 and red from about 52 to 58. The north pond validation points fell right inside that range, and the model got them right. The campus pond points had NIR around 35 and red around 68. That's outside the narrow range the model learned for water, but still darker than the asphalt training pixels, which start at an NIR of about 39.

So the campus pond sits in a gap the training data never covered. Every decision tree has to put its boundary somewhere in that gap, and with no examples there, the forest put it on the water side. Why is the campus pond different? Shallower water, suspended sediment or algae, and shade from the surrounding trees are all possible. The imagery alone can't settle it. Whatever the cause, the model could only learn water from the examples it was given, and all of them came from one pond.

That points to the fix, and to a trap:

- **The fix** is to add training examples from more kinds of water: other ponds, other depths, shaded edges. Then collect *new* independent validation points to test the updated model.
- **The trap** is to move the campus pond validation points into the training set and call the problem solved. The next section shows what that does to the score.

## Part 10: Class Areas

### Count Valid Pixels Only

A class-area summary is a categorical summary: count pixels per class and multiply by pixel area. Class codes are never averaged. The denominator is the number of **valid** pixels, not the full rectangle, so NoData never dilutes the percentages.

The image is in UTM zone 13N, so pixel area is in square meters. UTM isn't an equal-area projection, but its area distortion over a 1.7 km² AOI near the zone's center is a small fraction of a percent. For a large study area, use an equal-area CRS such as EPSG:5070, which is what Module 8 does.

In [ ]:
counts = np.bincount(classified[valid_mask], minlength=max(CLASS_IDS) + 1)
if counts[0] != 0:
    raise RuntimeError("A valid pixel was left unclassified.")

area_table = pd.DataFrame({
    'class_id': CLASS_IDS,
    'class_name': [CLASS_NAME[c] for c in CLASS_IDS],
    'pixels': [int(counts[c]) for c in CLASS_IDS],
})
area_table['area_m2'] = (area_table['pixels'] * cell_area_m2).round(0)
area_table['area_acres'] = (area_table['area_m2'] / 4046.8564224).round(1)
area_table['percent_of_valid_area'] = (100 * area_table['pixels'] / valid_mask.sum()).round(1)

total_area_acres = valid_mask.sum() * cell_area_m2 / 4046.8564224
print(f"Valid classified area: {total_area_acres:,.1f} acres "
      f"({area_table['percent_of_valid_area'].sum():.1f}% accounted for)")

area_path = OUTPUT_DIR / 'naip_landcover_class_area.csv'
area_export = area_table.assign(source_image=image_path.name,
                                acquisition_date=image_tags['acquisition_date'],
                                note='Screening classification; built/paved is not impervious area')
area_export.to_csv(area_path, index=False)
print(f"Saved: {area_path}")
area_table

Put these numbers in context before anyone quotes them:

- **They inherit every error from Part 8.** The campus pond counts as built/paved, which moves about an acre from water to built/paved. Some rangeland counts as bare soil.
- **Built/paved isn't impervious area.** Roof overhangs over lawns, cars parked on gravel, and shadows all end up here. Connected impervious area also needs drainage information.
- **Bare soil is a snapshot.** Most of it is active construction in June 2022 and will become houses, streets, or lawns.

A defensible way to report this is: *"About X percent of the AOI mapped as built/paved in June 2022 (screening classification, validated at 48 points; see limitations)."*

## Part 11: Context from Annual NLCD

Module 8 summarized Annual NLCD land cover for this watershed at 30 m. It's tempting to treat NLCD as the "right answer" and score the NAIP map against it. Don't. The two products differ in almost every way that matters:

| | NAIP screening map | Annual NLCD |
|---|---|---|
| Pixel size | 0.6 m | 30 m |
| Date | June 21, 2022 | 2025 (Module 8's course file) |
| Classes | 4 visual cover classes | 16 classes; "Developed" classes are defined by impervious *fraction* |
| Method | This lesson's Random Forest | National production model with its own validation |

NLCD is useful as **context**: does the overall pattern agree? The next cell reads the Module 8 course file, keeps cells whose centers fall inside the AOI, and summarizes them.

In [ ]:
NLCD_FILE = 'annual_nlcd_2025_land_cover.tif'
NLCD_NAMES = {
    11: 'Open Water', 21: 'Developed, Open Space', 22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity', 24: 'Developed, High Intensity', 31: 'Barren Land',
    41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest', 52: 'Shrub/Scrub',
    71: 'Grassland/Herbaceous', 81: 'Pasture/Hay', 82: 'Cultivated Crops',
    90: 'Woody Wetlands', 95: 'Emergent Herbaceous Wetlands',
}

try:
    nlcd_path, nlcd_source = get_course_file(NLCD_FILE, base_url=MODULE8_DATA_URL,
                                             module_folder='08_landuse-data')
    print(f"NLCD file: {nlcd_path} <- {nlcd_source}")
except requests.RequestException as error:
    nlcd_path = None
    print(f"The Module 8 NLCD file could not be downloaded ({error}). Skipping this part.")

if nlcd_path is not None:
    with rasterio.open(nlcd_path) as source:
        aoi_in_nlcd = gpd.GeoSeries([box(*AOI_BOUNDS)], crs=AOI_CRS).to_crs(source.crs)
        window = from_bounds(*aoi_in_nlcd.total_bounds, transform=source.transform)
        window = Window(math.floor(window.col_off), math.floor(window.row_off),
                        math.ceil(window.width) + 1, math.ceil(window.height) + 1)
        nlcd = source.read(1, window=window)
        nlcd_transform = source.window_transform(window)
        nlcd_nodata = source.nodata
        nlcd_colormap = source.colormap(1)

    # Keep 30 m cells whose centers fall inside the AOI polygon
    inside = geometry_mask(aoi_in_nlcd, out_shape=nlcd.shape, transform=nlcd_transform, invert=True)
    inside &= nlcd != nlcd_nodata
    nlcd_counts = pd.Series(nlcd[inside]).value_counts().sort_index()
    nlcd_table = pd.DataFrame({
        'nlcd_code': nlcd_counts.index,
        'nlcd_class': [NLCD_NAMES.get(code, str(code)) for code in nlcd_counts.index],
        'cells': nlcd_counts.values,
        'percent_of_aoi': (100 * nlcd_counts.values / inside.sum()).round(1),
    })
    display(nlcd_table)

    developed = nlcd_table.loc[nlcd_table['nlcd_code'].between(21, 24), 'percent_of_aoi'].sum()
    developed_med_high = nlcd_table.loc[nlcd_table['nlcd_code'].isin([23, 24]), 'percent_of_aoi'].sum()
    naip_built = area_table.loc[area_table['class_id'] == 3, 'percent_of_valid_area'].iloc[0]
    print(f"NLCD 2025, all Developed classes (21-24):   {developed:.1f}% of AOI cells")
    print(f"NLCD 2025, Developed medium + high (23-24): {developed_med_high:.1f}%")
    print(f"NAIP 2022 screening map, built/paved:       {naip_built:.1f}% of valid area")

In [ ]:
if nlcd_path is not None:
    codes_present = sorted(nlcd_counts.index)
    nlcd_display = np.ma.masked_where(~inside, np.searchsorted(codes_present, nlcd))
    nlcd_cmap = ListedColormap([np.array(nlcd_colormap[code][:3]) / 255 for code in codes_present])
    nlcd_extent = plotting_extent(nlcd, nlcd_transform)

    fig, axes = plt.subplots(1, 2, figsize=(16, 9))
    axes[0].imshow(np.ma.masked_equal(classified, 0), cmap=CLASS_COLORMAP, vmin=0.5, vmax=4.5,
                   extent=image_extent, interpolation='nearest')
    axes[0].set_title(f"NAIP {REFERENCE_YEAR} screening map (0.6 m, UTM 13N)")
    axes[0].legend(handles=[Patch(color=c, label=n) for n, c
                            in CLASSES[['class_name', 'color']].itertuples(index=False)],
                   loc='lower right', fontsize=9)
    axes[1].imshow(nlcd_display, cmap=nlcd_cmap, vmin=-0.5, vmax=len(codes_present) - 0.5,
                   extent=nlcd_extent, interpolation='nearest')
    aoi_in_nlcd.boundary.plot(ax=axes[1], color='black', linewidth=1)
    axes[1].set_title("Annual NLCD 2025 land cover (30 m, EPSG:5070)")
    axes[1].legend(handles=[Patch(color=np.array(nlcd_colormap[code][:3]) / 255,
                                  label=f"{code} {NLCD_NAMES.get(code, '')}") for code in codes_present],
                   loc='lower right', fontsize=8)
    for axis in axes:
        axis.set_xticks([])
        axis.set_yticks([])
    plt.tight_layout()
    plt.show()

The NLCD panel is in its own equal-area CRS, so the AOI appears as a slightly rotated rectangle. That's expected.

When this lesson was prepared, the broad pattern agreed: developed cover around the campus and the new streets, and grassland elsewhere. The percentages don't match, and they shouldn't be expected to. NLCD's Developed, Open Space class is mostly lawn and has a low impervious fraction. A 30 m cell with a road and some grass can be "developed" in NLCD and mostly vegetation in the NAIP map. The three-year gap also matters here, because construction kept going after June 2022.

**Question to ask your AI assistant:** *"NLCD says X percent developed and my NAIP classification says Y percent built/paved. List every reason the numbers could legitimately differ, and tell me which comparison, if any, would be fair."*

## Part 12: Practice Debugging - A Score That's Too Good to Be True

A colleague reviews your notebook and says: *"You're wasting 48 perfectly good labels on validation. Use all of them to train. More data, better model."* They send you the cell below, and it reports a much better score.

> **This cell contains an intentional bug.** It builds a separate `leaky_model`. Your real model, map, and validation results aren't changed.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING: THIS CELL CONTAINS AN INTENTIONAL BUG.
# It runs without an error. Your job is to explain what's wrong.
# ---------------------------------------------------------------
# "Use every label we have for training"
validation_pixels = image[:, point_rows, point_cols].T.astype('float32')
X_all = np.vstack([X_train, validation_pixels])
y_all = np.concatenate([y_train, results['reference_id'].to_numpy()])

leaky_model = RandomForestClassifier(
    n_estimators=N_TREES, min_samples_leaf=2, random_state=RANDOM_SEED, n_jobs=-1)
leaky_model.fit(X_all, y_all)

# "Check the new model on the validation points"
leaky_predictions = leaky_model.predict(validation_pixels)
leaky_accuracy = (leaky_predictions == results['reference_id'].to_numpy()).mean()

print(f"Colleague's validation accuracy: {leaky_accuracy:.1%}")
print(f"Accuracy from Part 8:            {overall_accuracy:.1%}")
print(f"Campus pond points now correct:  "
      f"{(leaky_predictions[campus_pond] == 4).sum()} of {campus_pond.sum()}")

### Your Task

1. **Trace the data.** Which pixels trained `leaky_model`? Which pixels was it scored on? Draw it out if that helps.
2. **Explain the jump.** Why did the score rise to near 100 percent? Did the model get better at mapping *other* ponds, or at remembering *these* 48 pixels?
3. **Decide what to report.** If you had to hand this map to the drainage reviewer tomorrow, which accuracy number would you put in the memo, and why?
4. **Propose a legitimate improvement.** How could you use the lesson from the campus pond to build a better model *and* still have an honest test?

**Try this with your AI assistant:**

*"My validation score jumped from about 83 percent to nearly 100 percent after a colleague's change. Here's the cell. Trace which labeled locations were used for training and which for checking the model. Are the validation samples independent? Suggest a proper split and two checks that would reveal leakage."*

Write your corrected approach below. It doesn't need to be long. A few lines that keep training and validation separate, plus a comment explaining what you'd do to improve the water class, is a complete answer.

In [ ]:
# YOUR FIX: keep validation independent.
#
# Hints:
# 1. The model you score must never have seen the pixels you score it on.
# 2. To improve water, what would you add to TRAINING, and what would you
#    need to collect afterwards to test the new model fairly?
# 3. Could the label check in Part 3 catch this mistake? Why or why not?

## Part 13: Record the Limitations

A classified map without its caveats will eventually be used for something it can't support. The next cell writes a plain-text summary that belongs next to the GeoTIFF: what went in, what came out, how well it validated, and what it shouldn't be used for. It fills in the numbers from this run, so they always match the map.

It also exports the point-by-point validation results and the confusion matrix.

In [ ]:
results_path = OUTPUT_DIR / 'naip_validation_results.csv'
confusion_path = OUTPUT_DIR / 'naip_confusion_matrix.csv'
summary_path = OUTPUT_DIR / 'naip_classification_summary.txt'
results.to_csv(results_path, index=False)
confusion.to_csv(confusion_path)

class_lines = '\n'.join(
    f"  {row['class']:<12} points {row['validation_points']:>2}   "
    f"producer's {row['producers_accuracy']:.2f}   user's {row['users_accuracy']:.2f}"
    for _, row in class_accuracy.fillna(0).iterrows())
area_lines = '\n'.join(
    f"  {row['class_name']:<12} {row['area_acres']:>7.1f} acres  ({row['percent_of_valid_area']:.1f}%)"
    for _, row in area_table.iterrows())
miss_lines = '\n'.join(
    f"  {row['point_id']}  {row['reference_class']} mapped as {row['predicted_class']} ({row['description']})"
    for _, row in misses.iterrows()) or '  none'

summary = f"""NAIP SCREENING LAND COVER CLASSIFICATION - SUMMARY AND LIMITATIONS
Created: {datetime.date.today().isoformat()}

SOURCE
  Image: {image_path.name}
  NAIP item {image_tags['source_item_id']}, acquired {image_tags['acquisition_date']}
  Source GSD {image_tags['source_gsd_m']} m; CRS {AOI_CRS}; public domain (USDA FSA)
  Teaching AOI {AOI_BOUNDS} (UTM 13N meters), inside HUC-12 101900090108

METHOD
  Classes: {', '.join(f'{k}={v}' for k, v in CLASS_NAME.items())}; 0=NoData
  Features: Red, Green, Blue, NIR (8-bit)
  Classifier: RandomForestClassifier, {N_TREES} trees, min_samples_leaf=2, seed {RANDOM_SEED}
  Training: {len(training)} polygons, {MAX_PIXELS_PER_CLASS} sampled pixels per class
  Labels: visual interpretation of the same image; review status: {training['review_status'].iloc[0]}

VALIDATION ({n_points} independent points, all >= {MIN_SEPARATION_M} m from training)
  Overall accuracy {overall_accuracy:.1%} (95% CI {ci_low:.1%} to {ci_high:.1%})
{class_lines}
  Misclassified points:
{miss_lines}

CLASS AREAS (valid pixels only)
{area_lines}

LIMITATIONS
  - Screening map for review only. Not a design dataset.
  - Built/paved is a visual cover class. It is not percent impervious or connected impervious area.
  - Water was trained from one pond; other water bodies may be mapped as built/paved.
  - Dry rangeland and bare soil are partly confused; shadows may map as water or built/paved.
  - Conditions are for {image_tags['acquisition_date']} only; construction areas are temporary.
  - The classifier should not be applied to another NAIP year without new labels and validation.
  - Validation uses {n_points} points; accuracy estimates carry wide uncertainty.
"""
summary_path.write_text(summary, encoding='utf-8')
print(summary)
print(f"Saved: {summary_path}, {results_path}, {confusion_path}")

## An Optional Advanced Path: Deep Learning Segmentation

You may have seen tools like [GeoAI](https://opengeoai.org/) and TorchGeo that train neural networks (for example, U-Net) to classify every pixel, or that offer pretrained models you can run on NAIP. They can capture shapes and textures a pixel-by-pixel Random Forest can't. They also bring GPUs, large downloads, many more labels, and many more ways for things to go quietly wrong.

If you explore them, carry this lesson's habits with you:

- **Know what the model's classes mean.** A pretrained scene model's "building" or "road" class wasn't defined for your project. Its labels aren't validated land cover for your AOI.
- **Keep an independent validation set.** A deep model can memorize training tiles even more easily than a Random Forest.
- **Check the license, the training data, and the size** of any pretrained weights before you rely on them.
- **Report the same things:** source date, class definitions, validation method, per-class accuracy, and limitations.

Useful starting points: [GeoAI classification API](https://opengeoai.org/classify/), [GeoAI land cover training example](https://opengeoai.org/examples/train_landcover_classification/), and [GeoAI AutoGeoModel examples](https://opengeoai.org/examples/AutoModel/).

## Engineering Cautions

1. **Validation must be independent.** Keep validation locations out of training, and far enough away that they aren't near-duplicates.
2. **Report per-class accuracy, not just the average.** Overall accuracy can look fine while one class fails completely.
3. **Report uncertainty.** A few dozen points can't support a precise accuracy claim.
4. **Area totals exclude NoData and never average class codes.**
5. **Built/paved isn't impervious area.** Don't feed it straight into a loss method or a curve number table.
6. **Don't transfer the model across years without new labels.** Season, sensor, and processing differences change the pixel values, as Lesson 2 showed.
7. **Don't claim numeric change between years** without separately validated, consistent classifications for both.

## Troubleshooting

| Problem | Likely cause | What to try |
|---|---|---|
| A label check fails | Labels edited, wrong CRS, or a point moved too close to training | Read the check table; fix the GeoJSON, then rerun Part 3 |
| "A validation point falls on NoData" | Point outside the image or in a masked area | Move or remove that point |
| All pixels are one class | Labels and image don't line up, or features weren't float | Replot Part 4; confirm both are in EPSG:26913 |
| Accuracy is very different from the lesson text | Different labels, seed, or package version | Compare the confusion matrix pattern, not just the number |
| NLCD part is skipped | Module 8 file couldn't be downloaded | Optional part; the rest of the lesson is unaffected |
| NLCD map looks rotated | It's in EPSG:5070 | Expected; don't reproject a categorical raster just for display |
| Prediction is slow | Very large image or many trees | Lower `N_TREES`, or increase `BLOCK_ROWS` if memory allows |

## Practice Exercises

### Exercise 1: Does NDVI Help?

Add NDVI as a fifth feature (compute it from float red and NIR, as in Lesson 2), retrain, repredict, and rerun the validation. Does overall accuracy change? Which points changed class?

When this lesson was prepared, adding NDVI raised the independent score noticeably, mostly by fixing campus pond points. Before you celebrate, answer three questions:

1. **Why might a ratio help here?** The campus pond is brighter than the north pond in every band. What happens to that brightness difference when you divide (NIR - Red) by (NIR + Red)?
2. **How many points actually changed?** With 48 validation points, is a change of three or four points convincing on its own? Compare the confidence intervals.
3. **Did you keep the test honest?** You chose NDVI *after* seeing the validation results. If you tried ten more features and kept the best one, what would that do to the meaning of the score?

In [ ]:
# EXERCISE 1: Add NDVI as a fifth feature
# Your code here.
#
# Steps:
# 1. ndvi = (nir - red) / (nir + red) using float arrays; set invalid cells to 0.
# 2. Stack it with the four bands for both the training samples and prediction.
# 3. Retrain, predict, and rebuild the confusion matrix with the same validation points.
# 4. Compare class_accuracy tables side by side.

### Exercise 2: Fix the Water Class the Right Way

Draw one new training polygon for water in the **campus pond**. Coordinates in EPSG:26913 around (518,070 to 518,110 E, 4,550,400 to 4,550,470 N) are a reasonable start; check them against the image. Retrain and repredict.

Now think carefully: the campus pond validation points are inside or next to your new polygon, so they're no longer independent. Remove them from the validation set, rerun the separation check, and report accuracy on the remaining points. What does that tell you about the value of collecting validation data from places that are never used for training?

In [ ]:
# EXERCISE 2: Add campus pond training data and keep validation honest
# Your code here.
#
# Steps:
# 1. Build a new polygon with shapely.geometry.box(...) in EPSG:26913, class_id 4.
# 2. Append it to `training` (gpd.GeoDataFrame, same CRS) and rerun Parts 5 to 8.
# 3. Drop validation points that are now within MIN_SEPARATION_M of training.
# 4. Report how many points remain and the new per-class accuracy.

### Challenge: Try the 2012 Image

Run the trained model on the 2012 image (Lesson 2 output, or `naip_2012_teaching_aoi.tif` from the course data folder). Resample it to 0.6 m first, or retrain on 1 m features. Look at the result. Is it believable? Why is applying a model trained on one acquisition to another a bad idea without new labels?

Suggested prompt:

*"Help me apply my 2022 Random Forest to the 2012 NAIP image of the same area. Then help me design a small labeling and validation plan for 2012 so I could compare the two years honestly."*

In [ ]:
# CHALLENGE: Apply the 2022 model to 2012
# Your code here.
#
# Steps:
# 1. Get naip_2012_teaching_aoi.tif with get_course_file().
# 2. Predict it in blocks with `model`, as in Part 7.
# 3. Plot the result next to the 2012 true color and describe what goes wrong.

### Download Your Outputs

In Colab, the next cell downloads the lesson outputs through your browser. Locally it prints their paths.

In [ ]:
output_files = [classified_path, map_path, results_path, confusion_path, area_path, summary_path]

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

for output_file in output_files:
    if not output_file.exists():
        raise FileNotFoundError(output_file)
    if colab_files is not None:
        colab_files.download(str(output_file))
    else:
        print(f"Output ready: {output_file.resolve()}")

## That's Module 9 Done!

You started this module with a catalog search and finished it with a validated, documented land cover screening map, all on a free Colab CPU. Along the way you checked coverage before date, grids before pixels, and independence before accuracy. In this lesson you also watched an 83 percent map fail completely on one pond, traced the failure to its cause, and saw how easily a leaky validation could have hidden it.

That last part is the one to hold onto. Most classification mistakes don't raise errors. They show up as a confident number in a memo.

### What You Can Do Now

- Find, download, and verify NAIP imagery for a small project area
- Compare two acquisitions on a common grid without mistaking artifacts for change
- Build a supervised classification from reviewed labels and validate it independently
- Report per-class accuracy with uncertainty, and write down what the map can't support

### Next Steps

- Pair the built/paved and vegetation classes with Module 7 soils and Module 8 NLCD in a curve number screening workflow, and keep the limitations attached.
- Try the optional advanced notebook `09_04_advanced_watershed_mosaic.ipynb` to build a full-watershed NAIP mosaic from raw web services.
- When project scale demands it, compare your screening results with local impervious layers, building footprints, and field observations.

### Resources

- [scikit-learn RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [GeoAI classification API](https://opengeoai.org/classify/)
- [GeoAI: train a land cover classification model](https://opengeoai.org/examples/train_landcover_classification/)
- [Annual NLCD product suite](https://www.usgs.gov/centers/eros/science/nlcd-product-suite)
- [NAIP on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/naip)